# UNIVERSIDAD EXTERNADO DE COLOMBIA · CIENCIA DE DATOS · 2026

## Modelo de doble valla (*hurdle*) para la duración de la estancia hospitalaria

**Predicción de la Estancia Hospitalaria con Aprendizaje Automático — SPARCS CY2021**

Autores: Daniel Vallejo Vecil · Juliana M.

---

Este cuaderno implementa y evalúa el esquema de modelado en dos fases que constituye la propuesta
metodológica del proyecto, comparando tres familias de algoritmos —regresión regularizada, Random
Forest y XGBoost con pérdida de Tweedie— bajo un protocolo idéntico, y contrastando el sistema de doble
valla frente al modelo de una sola etapa de cada familia.

## 1. Fundamento del esquema de doble valla

La duración de la estancia presenta una distribución estrictamente positiva y fuertemente asimétrica
(asimetría 5,82; curtosis 52,96), con una masa considerable concentrada en estancias muy breves y una
cola larga de casos prolongados. Un único modelo de regresión sobre esa distribución queda dominado por
el grueso de observaciones cortas y subestima sistemáticamente la cola, que es precisamente el segmento
de mayor relevancia para la gestión de camas.

Los modelos de doble valla, procedentes de la estadística de datos de conteo y del análisis de costos
en salud, separan el proceso que determina la **ocurrencia** de un evento del que determina su
**magnitud**, permitiendo que ambos dependan de covariables distintas sin forzar una especificación
común. Xu et al. (2022) trasladaron esta idea al aprendizaje automático para la predicción de estancia,
concluyendo que ningún modelo continuo único ofrecía un desempeño satisfactorio sobre la totalidad de
la distribución.

### 1.1 Definición de la valla

El corte se fija en 2 días, por correspondencia con el criterio regulatorio estadounidense. Desde
octubre de 2013, CMS considera apropiada la admisión como paciente internado cuando se espera que la
permanencia cruce dos medianoches. Bajo la fórmula de SPARCS —`LOS = (fecha egreso − fecha ingreso) + 1`—
cruzar dos medianoches equivale a `LOS >= 3`. Por tanto:

- **Estancia corta:** `LOS <= 2` (no alcanzó el referente normativo)
- **Estancia prolongada:** `LOS >= 3`

### 1.2 Tres formas de combinar las etapas

La literatura describe el esquema pero rara vez especifica cómo se recompone la predicción final. Este
cuaderno evalúa tres variantes de complejidad creciente, porque la elección no es un detalle de
implementación sino una decisión con consecuencias medibles:

| Variante | Componente corto | Componente largo | Recomposición |
|---|---|---|---|
| **A** | Constante (mediana) | Regresor sobre estancias largas | Enrutamiento duro por clase predicha |
| **B** | Regresor propio | Regresor sobre estancias largas | Enrutamiento duro por clase predicha |
| **C** | Regresor propio | Regresor sobre estancias largas | Esperanza ponderada por probabilidad |

La variante A reproduce la implementación más habitual y sirve de referencia. Su debilidad es doble:
asigna un valor idéntico a todos los episodios clasificados como cortos, sin discriminar dentro del
tramo, y propaga íntegramente los errores de clasificación, de modo que un episodio prolongado mal
clasificado recibe la constante y genera un error de gran magnitud.

La variante C recompone mediante la esperanza sobre ambos regímenes:

$$\hat{y}_i = \bigl(1 - P(\text{larga} \mid x_i)\bigr)\cdot \hat{y}^{\,corto}_i \; + \; P(\text{larga} \mid x_i)\cdot \hat{y}^{\,largo}_i$$

Al usar la probabilidad continua en lugar de la etiqueta discreta, un episodio con probabilidad 0,55 no
salta a la predicción completa del régimen largo, sino que queda ponderado entre ambos. Esto suaviza la
propagación de los errores de clasificación y elimina el escalón artificial que introduce el
enrutamiento duro.

## 2. Configuración y carga

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib, time, re

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().parent
GOLD_PATH = PROJECT_ROOT / "data" / "3_gold"
MODELOS_PATH = PROJECT_ROOT / "modelos"
TABLAS_PATH = PROJECT_ROOT / "tablas"
MODELOS_PATH.mkdir(parents=True, exist_ok=True)
TABLAS_PATH.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NUM = "08"


def guardar_tabla(tabla, nombre):
    if isinstance(tabla, pd.Series):
        tabla = tabla.to_frame()
    archivo = TABLAS_PATH / f"{NOTEBOOK_NUM}_{nombre}.csv"
    tabla.to_csv(archivo, encoding="utf-8-sig")
    print(f"Tabla guardada: {archivo.resolve()}")


# El split viene fijado desde 03_FeatureEngineering. NO se vuelve a particionar.
X_train = pd.read_parquet(GOLD_PATH / "X_train_sparcs.parquet")
X_test = pd.read_parquet(GOLD_PATH / "X_test_sparcs.parquet")
y_train = pd.read_parquet(GOLD_PATH / "y_train_sparcs.parquet")["Length of Stay"]
y_test = pd.read_parquet(GOLD_PATH / "y_test_sparcs.parquet")["Length of Stay"]

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")

X_train: (1672271, 189)   X_test: (418068, 189)


### 2.1 Saneamiento de los nombres de columna

Las columnas generadas por `get_dummies` conservan los caracteres de las categorías originales. Se
sanean para garantizar compatibilidad entre librerías y se conserva el mapa inverso, necesario para que
las gráficas del análisis de interpretabilidad muestren etiquetas legibles.

In [2]:
def sanear(nombres):
    """Convierte nombres de columna a caracteres seguros, evitando colisiones."""
    limpios, vistos = [], {}
    for n in nombres:
        s = re.sub(r"[^0-9a-zA-Z_]+", "_", str(n)).strip("_")
        if not s:
            s = "col"
        if s[0].isdigit():
            s = "v_" + s
        if s in vistos:
            vistos[s] += 1
            s = f"{s}__{vistos[s]}"
        else:
            vistos[s] = 0
        limpios.append(s)
    return limpios


NOMBRES_ORIGINALES = list(X_train.columns)
nombres_saneados = sanear(NOMBRES_ORIGINALES)
MAPA_NOMBRES = dict(zip(nombres_saneados, NOMBRES_ORIGINALES))

X_train.columns = nombres_saneados
X_test.columns = nombres_saneados

print(f"Columnas: {len(nombres_saneados)}  |  duplicadas tras saneamiento: "
      f"{len(nombres_saneados) - len(set(nombres_saneados))}")
joblib.dump(MAPA_NOMBRES, MODELOS_PATH / "08_mapa_nombres_columnas.joblib")

Columnas: 189  |  duplicadas tras saneamiento: 0


["c:\\Users\\julia\\OneDrive - Universidad Externado de Colombia\\Shortcuts\\Daniel Humberto Vallejo Vecil's files - Tesis\\SPARCS\\modelos\\08_mapa_nombres_columnas.joblib"]

### 2.2 Submuestreo para prototipado

Se recomienda una primera ejecución con `FRACCION = 0.20` para verificar el flujo, y la ejecución
definitiva con `FRACCION = 1.0`. El submuestreo afecta únicamente al entrenamiento; el conjunto de
prueba permanece completo, de modo que las métricas siguen siendo comparables con las de los cuadernos
anteriores.

In [3]:
FRACCION = 0.20   # poner 1.0 para la ejecución definitiva
SEMILLA = 42

if FRACCION < 1.0:
    idx = X_train.sample(frac=FRACCION, random_state=SEMILLA).index
    X_tr, y_tr = X_train.loc[idx], y_train.loc[idx]
else:
    X_tr, y_tr = X_train, y_train

print(f"Entrenamiento: {X_tr.shape[0]:,} filas   |   Prueba: {X_test.shape[0]:,} filas")

Entrenamiento: 334,454 filas   |   Prueba: 418,068 filas


## 3. Construcción de la valla

In [4]:
# Límite superior del tramo corto. Criterio de las dos medianoches (CMS).
UMBRAL_DIAS_CORTA = 2

y_tr_binaria = (y_tr > UMBRAL_DIAS_CORTA).astype(int)      # 1 = larga, 0 = corta
y_test_binaria = (y_test > UMBRAL_DIAS_CORTA).astype(int)

mask_corta_tr = y_tr_binaria == 0
mask_larga_tr = y_tr_binaria == 1

resumen_valla = pd.DataFrame({
    "tramo": ["Corta (<= 2 días)", "Prolongada (>= 3 días)"],
    "n_train": [int(mask_corta_tr.sum()), int(mask_larga_tr.sum())],
    "pct_train": [mask_corta_tr.mean() * 100, mask_larga_tr.mean() * 100],
    "pct_test": [(y_test_binaria == 0).mean() * 100, (y_test_binaria == 1).mean() * 100],
    "dias_medios": [y_tr[mask_corta_tr].mean(), y_tr[mask_larga_tr].mean()],
    "dias_mediana": [y_tr[mask_corta_tr].median(), y_tr[mask_larga_tr].median()],
}).set_index("tramo")

display(resumen_valla.round(2))
guardar_tabla(resumen_valla, "ResumenValla")

VALOR_CORTO_CONSTANTE = float(y_tr[mask_corta_tr].median())
CLASE_MAYORITARIA = max(mask_corta_tr.mean(), mask_larga_tr.mean())
print(f"\nConstante del tramo corto (variante A): {VALOR_CORTO_CONSTANTE}")
print(f"Línea base de exactitud (clase mayoritaria): {CLASE_MAYORITARIA:.4f}")

,n_train,pct_train,pct_test,dias_medios,dias_mediana
tramo,,,,,
Corta (<= 2 días),132076,39.49,39.53,1.56,2.0
Prolongada (>= 3 días),202378,60.51,60.47,8.48,5.0


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\08_ResumenValla.csv

Constante del tramo corto (variante A): 2.0
Línea base de exactitud (clase mayoritaria): 0.6051


## 4. Definición de las tres familias

Cada familia aporta cuatro componentes: un clasificador para la primera etapa, dos regresores de
magnitud —uno por tramo— y un regresor de una sola etapa que sirve de contraste. Los hiperparámetros se
mantienen equivalentes entre familias para que las diferencias observadas sean atribuibles al algoritmo
y no a su capacidad.

En la familia lineal, la primera etapa emplea regresión logística con penalización L2, contraparte
natural de Ridge para un desenlace binario.

In [5]:
PARAMS_RF = dict(n_estimators=200, max_depth=20, random_state=SEMILLA, n_jobs=-1)
PARAMS_XGB = dict(n_estimators=400, max_depth=8, learning_rate=0.05, subsample=0.8,
                  colsample_bytree=0.8, tree_method="hist", random_state=SEMILLA, n_jobs=-1)
POTENCIA_TWEEDIE = 1.5


def construir_familia(nombre):
    """Devuelve los cuatro componentes de una familia de algoritmos."""
    if nombre == "Ridge / Logística":
        return {
            "clasificador": LogisticRegression(max_iter=1000, n_jobs=-1),
            "reg_corto": Ridge(alpha=1.0),
            "reg_largo": Ridge(alpha=1.0),
            "reg_unico": Ridge(alpha=1.0),
        }
    if nombre == "Random Forest":
        return {
            "clasificador": RandomForestClassifier(**PARAMS_RF),
            "reg_corto": RandomForestRegressor(**PARAMS_RF),
            "reg_largo": RandomForestRegressor(**PARAMS_RF),
            "reg_unico": RandomForestRegressor(**PARAMS_RF),
        }
    if nombre == "XGBoost Tweedie":
        tw = dict(objective="reg:tweedie", tweedie_variance_power=POTENCIA_TWEEDIE)
        return {
            "clasificador": XGBClassifier(objective="binary:logistic",
                                          eval_metric="logloss", **PARAMS_XGB),
            "reg_corto": XGBRegressor(**{**PARAMS_XGB, **tw}),
            "reg_largo": XGBRegressor(**{**PARAMS_XGB, **tw}),
            "reg_unico": XGBRegressor(**{**PARAMS_XGB, **tw}),
        }
    raise ValueError(nombre)


FAMILIAS = ["Ridge / Logística", "Random Forest", "XGBoost Tweedie"]
print("Familias a evaluar:", FAMILIAS)

Familias a evaluar: ['Ridge / Logística', 'Random Forest', 'XGBoost Tweedie']


## 5. Primera etapa: clasificación corta / prolongada

Se reporta AUC-ROC además de las métricas de umbral, porque las probabilidades —y no solo la etiqueta—
alimentan la recomposición de la variante C. La exactitud debe interpretarse contra la línea base de la
clase mayoritaria calculada en la Sección 3.

In [6]:
clasificadores, probas_test, filas_clf = {}, {}, []

for fam in FAMILIAS:
    comps = construir_familia(fam)
    clf = comps["clasificador"]
    t0 = time.time()
    clf.fit(X_tr, y_tr_binaria)

    pred_clase = clf.predict(X_test)
    proba_larga = clf.predict_proba(X_test)[:, 1]

    clasificadores[fam] = clf
    probas_test[fam] = proba_larga

    filas_clf.append({
        "familia": fam,
        "accuracy": accuracy_score(y_test_binaria, pred_clase),
        "precision": precision_score(y_test_binaria, pred_clase, zero_division=0),
        "recall": recall_score(y_test_binaria, pred_clase, zero_division=0),
        "f1": f1_score(y_test_binaria, pred_clase, zero_division=0),
        "auc_roc": roc_auc_score(y_test_binaria, proba_larga),
        "segundos": round(time.time() - t0, 1),
    })
    print(f"{fam:20s}  acc={filas_clf[-1]['accuracy']:.4f}  F1={filas_clf[-1]['f1']:.4f}  "
          f"AUC={filas_clf[-1]['auc_roc']:.4f}  ({filas_clf[-1]['segundos']}s)")

clf_df = pd.DataFrame(filas_clf)
clf_df["ganancia_sobre_linea_base"] = clf_df["accuracy"] - CLASE_MAYORITARIA
display(clf_df.round(4))
guardar_tabla(clf_df, "Etapa1Clasificacion")

Ridge / Logística     acc=0.7513  F1=0.7986  AUC=0.8142  (61.9s)
Random Forest         acc=0.7656  F1=0.8146  AUC=0.8343  (97.4s)
XGBoost Tweedie       acc=0.7748  F1=0.8189  AUC=0.8468  (36.1s)


,familia,accuracy,precision,recall,f1,auc_roc,segundos,ganancia_sobre_linea_base
0,Ridge / Logística,0.7513,0.7824,0.8154,0.7986,0.8142,61.9,0.1462
1,Random Forest,0.7656,0.7809,0.8513,0.8146,0.8343,97.4,0.1605
2,XGBoost Tweedie,0.7748,0.7971,0.8419,0.8189,0.8468,36.1,0.1697


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\08_Etapa1Clasificacion.csv


## 6. Segunda etapa: regresores de magnitud por tramo

Cada familia ajusta dos regresores independientes, uno por régimen. El del tramo corto opera sobre un
rango de dos valores posibles, de modo que su contribución al error global es pequeña pero no nula: la
variante A lo sustituye por una constante y con ello renuncia a toda discriminación dentro del tramo.

In [7]:
regresores = {}

for fam in FAMILIAS:
    comps = construir_familia(fam)
    print(f"Entrenando regresores de magnitud: {fam} ...")

    rc = comps["reg_corto"]
    rc.fit(X_tr[mask_corta_tr.to_numpy()], y_tr[mask_corta_tr.to_numpy()])

    rl = comps["reg_largo"]
    rl.fit(X_tr[mask_larga_tr.to_numpy()], y_tr[mask_larga_tr.to_numpy()])

    ru = comps["reg_unico"]
    ru.fit(X_tr, y_tr)

    regresores[fam] = {"corto": rc, "largo": rl, "unico": ru}

print("\nRegresores entrenados para las tres familias.")

Entrenando regresores de magnitud: Ridge / Logística ...
Entrenando regresores de magnitud: Random Forest ...
Entrenando regresores de magnitud: XGBoost Tweedie ...

Regresores entrenados para las tres familias.


### 6.1 Desempeño condicional del regresor del tramo prolongado

Se evalúa sobre los episodios **realmente** prolongados del conjunto de prueba, aislando el desempeño
de la segunda etapa del efecto de los errores de clasificación. Esta cifra no es comparable con las
métricas globales de las secciones siguientes, porque se calcula sobre un subconjunto distinto.

In [8]:
mask_larga_test = (y_test_binaria == 1).to_numpy()
filas_cond = []

for fam in FAMILIAS:
    p = np.clip(regresores[fam]["largo"].predict(X_test[mask_larga_test]), 1, None)
    yr = y_test.to_numpy()[mask_larga_test]
    filas_cond.append({
        "familia": fam,
        "n": int(mask_larga_test.sum()),
        "MAE": mean_absolute_error(yr, p),
        "RMSE": float(np.sqrt(mean_squared_error(yr, p))),
        "R2": r2_score(yr, p),
    })

cond_df = pd.DataFrame(filas_cond)
display(cond_df.round(4))
guardar_tabla(cond_df, "Etapa2Condicional")

,familia,n,MAE,RMSE,R2
0,Ridge / Logística,252809,4.2557,7.8964,0.3442
1,Random Forest,252809,3.9593,7.7108,0.3746
2,XGBoost Tweedie,252809,3.8211,7.5180,0.4055


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\08_Etapa2Condicional.csv


## 7. Recomposición: las tres variantes frente al modelo único

Todas las métricas se calculan sobre el conjunto de prueba completo y en escala de días, de modo que
son directamente comparables entre variantes y con los cuadernos anteriores.

In [9]:
def metricas(y_real, y_pred):
    y_pred = np.clip(y_pred, 1, None)
    return {"MAE": mean_absolute_error(y_real, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_real, y_pred))),
            "R2": r2_score(y_real, y_pred)}


resultados, predicciones = [], {}
y_test_np = y_test.to_numpy()

for fam in FAMILIAS:
    proba = probas_test[fam]
    clase = (proba >= 0.5).astype(int)

    p_corto = np.clip(regresores[fam]["corto"].predict(X_test), 1, None)
    p_largo = np.clip(regresores[fam]["largo"].predict(X_test), 1, None)
    p_unico = np.clip(regresores[fam]["unico"].predict(X_test), 1, None)

    # Variante A: constante + enrutamiento duro
    pred_a = np.where(clase == 1, p_largo, VALOR_CORTO_CONSTANTE)
    # Variante B: dos regresores + enrutamiento duro
    pred_b = np.where(clase == 1, p_largo, p_corto)
    # Variante C: dos regresores + esperanza ponderada
    pred_c = (1 - proba) * p_corto + proba * p_largo

    for etiqueta, pred in [
        ("Modelo único (una etapa)", p_unico),
        ("Doble valla A — constante + duro", pred_a),
        ("Doble valla B — dos regresores + duro", pred_b),
        ("Doble valla C — dos regresores + esperanza", pred_c),
    ]:
        resultados.append({"familia": fam, "estrategia": etiqueta, **metricas(y_test, pred)})
        predicciones[(fam, etiqueta)] = np.clip(pred, 1, None)

res_df = pd.DataFrame(resultados)
display(res_df.round(4))
guardar_tabla(res_df, "ComparacionDobleValla")

pivote = res_df.pivot(index="estrategia", columns="familia", values="R2")
print("\nR2 por estrategia y familia:")
display(pivote.round(4))
guardar_tabla(pivote, "PivoteR2DobleValla")

,familia,estrategia,MAE,RMSE,R2
0,Ridge / Logística,Modelo único (una etapa),3.2710,6.5180,0.3836
1,Ridge / Logística,Doble valla A — constante + duro,3.4680,6.5844,0.3710
2,Ridge / Logística,Doble valla B — dos regresores + duro,3.5121,6.5987,0.3683
3,Ridge / Logística,Doble valla C — dos regresores + esperanza,3.1950,6.4818,0.3904
4,Random Forest,Modelo único (una etapa),3.0134,6.3108,0.4222
5,Random Forest,Doble valla A — constante + duro,3.2818,6.4297,0.4002
6,Random Forest,Doble valla B — dos regresores + duro,3.3003,6.4380,0.3986
7,Random Forest,Doble valla C — dos regresores + esperanza,3.0065,6.3066,0.4229
8,XGBoost Tweedie,Modelo único (una etapa),2.9180,6.1640,0.4487
9,XGBoost Tweedie,Doble valla A — constante + duro,3.1331,6.2291,0.4370


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\08_ComparacionDobleValla.csv

R2 por estrategia y familia:


familia,Random Forest,Ridge / Logística,XGBoost Tweedie
estrategia,,,
Doble valla A — constante + duro,0.4002,0.3710,0.4370
Doble valla B — dos regresores + duro,0.3986,0.3683,0.4351
Doble valla C — dos regresores + esperanza,0.4229,0.3904,0.4489
Modelo único (una etapa),0.4222,0.3836,0.4487


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\08_PivoteR2DobleValla.csv


La lectura relevante es la comparación **fila a fila dentro de cada familia**: si la variante C supera a
la A, la ganancia proviene de la recomposición y no del algoritmo. Y la comparación de cada variante
contra el modelo único de su propia familia determina si la separación en dos regímenes aporta algo por
encima de modelar la distribución completa de una sola vez.

## 8. Descomposición del error por celda de la matriz de confusión

Este diagnóstico explica por qué un clasificador con buena exactitud puede producir un sistema con
desempeño global mediocre. Los falsos negativos —episodios prolongados clasificados como cortos— suelen
representar una fracción pequeña de los registros y una fracción desproporcionada del error cuadrático
total, porque reciben una predicción del régimen equivocado.

In [11]:
FAMILIA_DIAG = "XGBoost Tweedie"
VARIANTES_DIAG = ["Doble valla A — constante + duro",
                  "Doble valla C — dos regresores + esperanza"]

clase_pred = (probas_test[FAMILIA_DIAG] >= 0.5).astype(int)
real_bin = y_test_binaria.to_numpy()

celda = np.select(
    [(real_bin == 0) & (clase_pred == 0),
     (real_bin == 0) & (clase_pred == 1),
     (real_bin == 1) & (clase_pred == 0),
     (real_bin == 1) & (clase_pred == 1)],
    ["VN: corta->corta", "FP: corta->larga", "FN: larga->corta", "VP: larga->larga"],
    default="sin_clasificar",
)

if (celda == "sin_clasificar").any():
    print(f"AVISO: {(celda == 'sin_clasificar').sum()} registros sin clasificar. Revisar nulos.")

filas_diag = []
for var in VARIANTES_DIAG:
    pred = predicciones[(FAMILIA_DIAG, var)]
    d = pd.DataFrame({"celda": celda, "real": y_test_np, "pred": pred})
    d["err_abs"] = (d.real - d.pred).abs()
    d["err_sq"] = (d.real - d.pred) ** 2
    g = d.groupby("celda").agg(n=("real", "size"), real_medio=("real", "mean"),
                               pred_media=("pred", "mean"), MAE=("err_abs", "mean"),
                               SSE=("err_sq", "sum"))
    g["pct_n"] = g.n / len(d) * 100
    g["pct_SSE"] = g.SSE / g.SSE.sum() * 100
    g["variante"] = var
    filas_diag.append(g.reset_index())

diag_df = pd.concat(filas_diag, ignore_index=True)
display(diag_df.round(2))
guardar_tabla(diag_df, "DescomposicionErrorPorCelda")

cm = confusion_matrix(y_test_binaria, clase_pred)
cm_df = pd.DataFrame(cm, index=["Real corta", "Real larga"],
                     columns=["Pred corta", "Pred larga"])
display(cm_df)
guardar_tabla(cm_df, "MatrizConfusionEtapa1")

,celda,n,real_medio,pred_media,MAE,SSE,pct_n,pct_SSE,variante
0,FN: larga->corta,39972,4.04,2.00,2.04,418646.00,9.56,2.58,Doble valla A — constante + duro
1,FP: corta->larga,54185,1.62,6.36,4.74,1695174.17,12.96,10.45,Doble valla A — constante + duro
2,VN: corta->corta,111074,1.53,2.00,0.47,52414.00,26.57,0.32,Doble valla A — constante + duro
3,VP: larga->larga,212837,9.27,9.04,4.32,14055270.63,50.91,86.65,Doble valla A — constante + duro
4,FN: larga->corta,39972,4.04,2.48,1.59,336963.32,9.56,2.12,Doble valla C — dos regresores + esperanza
5,FP: corta->larga,54185,1.62,5.10,3.48,1084161.69,12.96,6.83,Doble valla C — dos regresores + esperanza
6,VN: corta->corta,111074,1.53,2.16,0.70,93722.17,26.57,0.59,Doble valla C — dos regresores + esperanza
7,VP: larga->larga,212837,9.27,8.10,4.17,14365097.22,50.91,90.46,Doble valla C — dos regresores + esperanza


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\08_DescomposicionErrorPorCelda.csv


,Pred corta,Pred larga
Real corta,111074,54185
Real larga,39972,212837


Tabla guardada: C:\Users\julia\OneDrive - Universidad Externado de Colombia\Shortcuts\Daniel Humberto Vallejo Vecil's files - Tesis\SPARCS\tablas\08_MatrizConfusionEtapa1.csv


La columna `pct_SSE` indica qué fracción del error cuadrático total aporta cada celda. Si los falsos
negativos concentran un porcentaje de error muy superior a su porcentaje de registros, el cuello de
botella del sistema está en la sensibilidad del clasificador y no en la calidad de los regresores.

## 9. Desempeño por tramo de estancia

In [12]:
TRAMOS = [(1, 2, "1-2 días"), (3, 6, "3-6 días"), (7, 13, "7-13 días"),
          (14, 29, "14-29 días"), (30, 119, "30-119 días"), (120, 120, "120+ (censurado)")]

CANDIDATOS = [(FAMILIA_DIAG, "Modelo único (una etapa)"),
              (FAMILIA_DIAG, "Doble valla A — constante + duro"),
              (FAMILIA_DIAG, "Doble valla C — dos regresores + esperanza")]

filas = []
for clave in CANDIDATOS:
    pred = predicciones[clave]
    for lo, hi, etiqueta in TRAMOS:
        m = (y_test_np >= lo) & (y_test_np <= hi)
        if m.sum() == 0:
            continue
        filas.append({"estrategia": clave[1], "tramo": etiqueta, "n": int(m.sum()),
                      "pct_test": m.mean() * 100,
                      "real_medio": float(y_test_np[m].mean()),
                      "pred_media": float(pred[m].mean()),
                      "MAE": mean_absolute_error(y_test_np[m], pred[m])})

tramos_df = pd.DataFrame(filas)
display(tramos_df.round(2))
# guardar_tabla(tramos_df, "DesempenoPorTramo")

,estrategia,tramo,n,pct_test,real_medio,pred_media,MAE
0,Modelo único (una etapa),1-2 días,165259,39.53,1.56,3.12,1.60
1,Modelo único (una etapa),3-6 días,152033,36.37,4.06,4.93,1.84
2,Modelo único (una etapa),7-13 días,64491,15.43,9.16,8.34,3.43
3,Modelo único (una etapa),14-29 días,28338,6.78,18.94,12.57,8.03
4,Modelo único (una etapa),30-119 días,7658,1.83,46.35,21.76,26.37
5,Modelo único (una etapa),120+ (censurado),289,0.07,120.00,37.47,82.53
6,Doble valla A — constante + duro,1-2 días,165259,39.53,1.56,3.43,1.87
7,Doble valla A — constante + duro,3-6 días,152033,36.37,4.06,5.59,2.40
8,Doble valla A — constante + duro,7-13 días,64491,15.43,9.16,9.17,3.20
9,Doble valla A — constante + duro,14-29 días,28338,6.78,18.94,13.35,7.42


## 10. Selección del modelo final y preparación del análisis de interpretabilidad

Se persiste el componente que será objeto del análisis SHAP. Si la variante ganadora es de doble valla,
la interpretación debe realizarse **por separado sobre cada componente**: el clasificador de la primera
etapa responde a qué factores determinan que un episodio supere el umbral, y el regresor de la segunda,
a qué factores gobiernan su magnitud una vez superado. Son dos preguntas distintas y las atribuciones no
son intercambiables.

In [13]:
mejor = res_df.sort_values("MAE").iloc[0]
print(f"Menor MAE:  {mejor['familia']} — {mejor['estrategia']}  (MAE={mejor['MAE']:.3f} días)")
mejor_r2 = res_df.sort_values("R2", ascending=False).iloc[0]
print(f"Mayor R2:   {mejor_r2['familia']} — {mejor_r2['estrategia']}  (R2={mejor_r2['R2']:.4f})")

joblib.dump(clasificadores[FAMILIA_DIAG], MODELOS_PATH / "08_etapa1_clasificador.joblib")
joblib.dump(regresores[FAMILIA_DIAG]["largo"], MODELOS_PATH / "08_etapa2_reg_largo.joblib")
joblib.dump(regresores[FAMILIA_DIAG]["corto"], MODELOS_PATH / "08_etapa2_reg_corto.joblib")
joblib.dump(regresores[FAMILIA_DIAG]["unico"], MODELOS_PATH / "08_modelo_unico.joblib")

# Muestra estratificada del conjunto de prueba para el cuaderno de SHAP
muestra_shap = X_test.sample(n=min(50_000, len(X_test)), random_state=SEMILLA)
muestra_shap.to_parquet(MODELOS_PATH / "08_muestra_shap.parquet")
y_test.loc[muestra_shap.index].to_frame().to_parquet(MODELOS_PATH / "08_muestra_shap_y.parquet")

print("\nModelos y muestra para SHAP guardados en /modelos.")

Menor MAE:  XGBoost Tweedie — Doble valla C — dos regresores + esperanza  (MAE=2.914 días)
Mayor R2:   XGBoost Tweedie — Doble valla C — dos regresores + esperanza  (R2=0.4489)

Modelos y muestra para SHAP guardados en /modelos.


## 11. Refinamiento pendiente: alineación de la población de entrenamiento

El regresor del tramo prolongado se entrena sobre los episodios **realmente** prolongados, pero en
inferencia se aplica a los episodios **predichos** como prolongados, conjunto que incluye falsos
positivos con estancia real de uno o dos días. El regresor nunca observó un episodio corto durante su
ajuste, por lo que su predicción mínima posible se sitúa cerca del extremo inferior de la distribución
de estancias largas: existe un desajuste entre la población de entrenamiento y la de aplicación.

La corrección consiste en entrenar el regresor sobre los episodios que el clasificador predice como
prolongados en el propio conjunto de entrenamiento, empleando predicciones fuera de pliegue para evitar
que el clasificador evalúe observaciones que ya vio. La variante C mitiga en parte este efecto al
ponderar por probabilidad, pero no lo elimina.

Se implementa a continuación como variante opcional, desactivada por defecto por su costo.

In [14]:
EJECUTAR_VARIANTE_D = False   # requiere validación cruzada adicional

if EJECUTAR_VARIANTE_D:
    from sklearn.model_selection import cross_val_predict, StratifiedKFold

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
    proba_oof = cross_val_predict(construir_familia(FAMILIA_DIAG)["clasificador"],
                                  X_tr, y_tr_binaria, cv=skf, method="predict_proba")[:, 1]
    mask_pred_larga_tr = proba_oof >= 0.5
    print(f"Episodios predichos como prolongados en entrenamiento: {mask_pred_larga_tr.sum():,}")

    rl_d = construir_familia(FAMILIA_DIAG)["reg_largo"]
    rl_d.fit(X_tr[mask_pred_larga_tr], y_tr[mask_pred_larga_tr])

    p_largo_d = np.clip(rl_d.predict(X_test), 1, None)
    p_corto_d = np.clip(regresores[FAMILIA_DIAG]["corto"].predict(X_test), 1, None)
    proba = probas_test[FAMILIA_DIAG]
    pred_d = (1 - proba) * p_corto_d + proba * p_largo_d

    fila_d = pd.DataFrame([{"familia": FAMILIA_DIAG,
                            "estrategia": "Doble valla D — población alineada + esperanza",
                            **metricas(y_test, pred_d)}])
    display(fila_d.round(4))
    guardar_tabla(fila_d, "VarianteDPoblacionAlineada")
else:
    print("Variante D desactivada. Cambiar EJECUTAR_VARIANTE_D = True para ejecutarla.")

Variante D desactivada. Cambiar EJECUTAR_VARIANTE_D = True para ejecutarla.


## 12. Conclusiones del cuaderno

Elementos a documentar en el Capítulo 4:

1. **Comparación entre variantes de recomposición.** Si C supera a A dentro de cada familia, la ganancia
   procede del método de combinación y no del algoritmo. Es un aporte metodológico propio, dado que la
   literatura describe el esquema de doble valla sin especificar cómo recompone la predicción.
2. **Doble valla frente a modelo único.** El resultado, sea cual sea, responde directamente a la
   pregunta de investigación. Si el modelo único gana incluso con la mejor recomposición, la conclusión
   coincide con Xu et al. (2022) sobre la dificultad intrínseca de predecir las estancias prolongadas.
3. **Las métricas de la primera etapa contra la línea base.** Una exactitud alta sobre una clase
   mayoritaria dominante puede ser engañosa; el AUC-ROC y la ganancia sobre la línea base son las cifras
   a reportar.
4. **La descomposición del error por celda.** Es el diagnóstico que explica el comportamiento del
   sistema y, muy probablemente, el resultado más informativo del cuaderno.
5. **La tabla por tramos.** Documenta en qué región del desenlace el sistema produce estimaciones
   confiables y en cuál no, que es precisamente el argumento con que Xu et al. justifican el esquema.

**Tablas exportadas:** `08_ResumenValla`, `08_Etapa1Clasificacion`, `08_Etapa2Condicional`,
`08_ComparacionDobleValla`, `08_PivoteR2DobleValla`, `08_DescomposicionErrorPorCelda`,
`08_MatrizConfusionEtapa1`, `08_DesempenoPorTramo`.